In [123]:
import pandas as pd
from pathlib import Path

In [124]:
def load_csvs_from_dir(directory, pattern="*.csv", concat=True, add_filename_col=True, sort=True, **read_csv_kwargs):
    """
    Load CSV files from `directory`.

    Args:
    - directory (str or Path): path to folder containing CSV files.
    - pattern (str): glob pattern to match files (default "*.csv").
    - concat (bool): if True, return a single concatenated DataFrame; if False, return a list of DataFrames.
    - add_filename_col (bool): if True, add a column '_source_file' with the filename for each row.
    - sort (bool): if True, sort files by name before loading.
    - **read_csv_kwargs: passed to `pd.read_csv` (e.g., parse_dates=["time"]).

    Returns:
    - pandas.DataFrame or list[pandas.DataFrame]
    """
    p = Path(directory)
    files = list(p.glob(pattern))
    if sort:
        files = sorted(files)
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, **read_csv_kwargs)
        except Exception as e:
            # re-raise with context
            raise RuntimeError(f"Failed to read {f}: {e}") from e
        if add_filename_col:
            df["_source_file"] = f.name
        dfs.append(df)
    if concat:
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        return dfs

In [125]:
powerwall_data_path="../tesla-solar-download/download/2733356/energy"
df_raw = load_csvs_from_dir(powerwall_data_path, add_filename_col=False)

In [126]:
df_raw.count()

timestamp                                93944
solar_energy_exported                    93944
grid_energy_imported                     93944
grid_energy_exported_from_solar          93944
grid_energy_exported_from_battery        93944
battery_energy_exported                  93944
battery_energy_imported_from_grid        93944
battery_energy_imported_from_solar       93944
consumer_energy_imported_from_grid       93944
consumer_energy_imported_from_solar      93944
consumer_energy_imported_from_battery    93944
raw_timestamp                            93944
total_battery_charge                         7
total_grid_energy_exported               18170
total_home_usage                         93916
total_battery_discharge                  34418
dtype: int64

In [127]:
df_raw.head()

,timestamp,solar_energy_exported,grid_energy_imported,grid_energy_exported_from_solar,grid_energy_exported_from_battery,battery_energy_exported,battery_energy_imported_from_grid,battery_energy_imported_from_solar,consumer_energy_imported_from_grid,consumer_energy_imported_from_solar,consumer_energy_imported_from_battery,raw_timestamp,total_battery_charge,total_grid_energy_exported,total_home_usage,total_battery_discharge
0,2020-07-31 10:00:00,0.0,0.0,0.0,0.0,0,20.0,0.0,0.0,0.0,0.0,2020-07-31T10:00:00+02:00,20.0,127.0,NaN,NaN
1,2020-07-31 10:30:00,513.0,0.0,248.0,0.0,0,0.0,100.0,0.0,165.0,0.0,2020-07-31T10:30:00+02:00,100.0,248.0,NaN,NaN
2,2020-07-31 11:00:00,594.0,0.0,360.0,0.0,0,0.0,100.0,0.0,134.0,0.0,2020-07-31T11:00:00+02:00,100.0,360.0,NaN,NaN
3,2020-07-31 11:30:00,1286.0,0.0,359.0,0.0,0,0.0,780.0,0.0,147.0,0.0,2020-07-31T11:30:00+02:00,780.0,359.0,NaN,NaN
4,2020-07-31 12:00:00,2038.0,1.0,0.0,0.0,0,0.0,1050.0,1.0,988.0,0.0,2020-07-31T12:00:00+02:00,1050.0,NaN,NaN,NaN


In [128]:
df=pd.DataFrame()
timestamps = pd.to_datetime(df_raw["timestamp"], dayfirst=False, errors="raise")
#df["date"]=timestamps.dt.date
#df["time"]=timestamps.dt.time
df["timestamp"]=timestamps
df["month"]=timestamps.dt.month

In [129]:
df.head()

,timestamp,month
0,2020-07-31 10:00:00,7
1,2020-07-31 10:30:00,7
2,2020-07-31 11:00:00,7
3,2020-07-31 11:30:00,7
4,2020-07-31 12:00:00,7


In [130]:
df['solar_energy_exported'] = df_raw['solar_energy_exported']
df['total_home_usage']=df_raw['consumer_energy_imported_from_grid']+df_raw['consumer_energy_imported_from_solar']+df_raw['consumer_energy_imported_from_battery']

In [131]:
# compute solar surplus (at minimum it must be 0)
df['solar_surplus']=df['solar_energy_exported'] - df['total_home_usage']
df.solar_surplus=df.solar_surplus.apply(lambda x: max(0,x))
df.head()

,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus
0,2020-07-31 10:00:00,7,0.0,0.0,0.0
1,2020-07-31 10:30:00,7,513.0,165.0,348.0
2,2020-07-31 11:00:00,7,594.0,134.0,460.0
3,2020-07-31 11:30:00,7,1286.0,147.0,1139.0
4,2020-07-31 12:00:00,7,2038.0,989.0,1049.0


In [132]:
df.count()

timestamp                93944
month                    93944
solar_energy_exported    93944
total_home_usage         93944
solar_surplus            93944
dtype: int64

In [133]:
df[(df.solar_surplus>0)].count()

timestamp                31227
month                    31227
solar_energy_exported    31227
total_home_usage         31227
solar_surplus            31227
dtype: int64

In [134]:
# check NaN values
df.dropna().count()

timestamp                93944
month                    93944
solar_energy_exported    93944
total_home_usage         93944
solar_surplus            93944
dtype: int64

In [135]:
df['solar_surplus'].describe()

count    93944.000000
mean       247.711310
std        476.339903
min          0.000000
25%          0.000000
50%          0.000000
75%        250.503569
max       2456.356447
Name: solar_surplus, dtype: float64

In [136]:
# keep only rows where solar_surplus is positive
df = df[(df.solar_surplus>0)]

In [137]:
df.count()

timestamp                31227
month                    31227
solar_energy_exported    31227
total_home_usage         31227
solar_surplus            31227
dtype: int64

In [ ]:
df.count()

In [138]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests

In [139]:
import pandas as pd
import requests_cache
from retry_requests import retry
import openmeteo_requests

In [186]:
def get_historical_weather(latitude: float, longitude: float, start_date: str, end_date=None, timezone="Europe/Berlin"):
    
    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)
    
    # Make sure all required weather variables are listed here
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
    	"latitude": {latitude},
    	"longitude": {longitude},
    	"start_date": {start_date},
    	"end_date": { (end_date if end_date is not None else start_date) },
    	#"daily": ["sunrise", "sunset"],
    	"hourly": ["weather_code", "cloud_cover"],
    	"timezone": {timezone},
    }
    responses = openmeteo.weather_api(url, params=params)
    
    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    #print(response)
    hourly_weather_code = hourly.Variables(0).ValuesAsNumpy()
    hourly_cloud_cover = hourly.Variables(1).ValuesAsNumpy()
    
    hourly_data = {"timestamp": pd.date_range(
    	start = pd.to_datetime(hourly.Time(), unit = "s", utc = False),
    	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = False),
    	freq = pd.Timedelta(seconds = hourly.Interval()),
    	inclusive = "left"
    )}
    
    hourly_data["weather_code"] = hourly_weather_code
    hourly_data["cloud_cover"] = hourly_cloud_cover
    
    hourly_df = pd.DataFrame(data = hourly_data)
    #print("\nHourly data\n", hourly_dataframe)

    #ts = pd.to_datetime(hourly_df["date_time"], dayfirst=False, errors="raise")
    # hourly_df["date"]=ts.dt.date
    # hourly_df["time"]=ts.dt.time
    
    return hourly_df
    
    # Process daily data. The order of variables needs to be the same as requested.
    #daily = response.Daily()
    #daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
    #daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()
    
    #daily_data = {"date": pd.date_range(
    # 	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
    #	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
    #	freq = pd.Timedelta(seconds = daily.Interval()),
    #	inclusive = "left"
    #)}
    
    #daily_data["sunrise"] = daily_sunrise
    #daily_data["sunset"] = daily_sunset
    
    #daily_dataframe = pd.DataFrame(data = daily_data)
    #print("\nDaily data\n", daily_dataframe)
    

In [187]:
# Solar plant Coordinates
latitude=45.73253954365124
longitude=9.714445021009563
#timezone="Europe/Berlin"
hourly_df = get_historical_weather(latitude, longitude, '2025-12-15')

In [188]:
print("\nHourly data\n", hourly_df)


Hourly data
              timestamp  weather_code  cloud_cover
0  2025-12-14 23:00:00           0.0          0.0
1  2025-12-15 00:00:00           0.0          0.0
2  2025-12-15 01:00:00           0.0          0.0
3  2025-12-15 02:00:00           0.0          1.0
4  2025-12-15 03:00:00           0.0         10.0
5  2025-12-15 04:00:00           0.0          0.0
6  2025-12-15 05:00:00           0.0          1.0
7  2025-12-15 06:00:00           1.0         49.0
8  2025-12-15 07:00:00           1.0         22.0
9  2025-12-15 08:00:00           3.0         91.0
10 2025-12-15 09:00:00           1.0         30.0
11 2025-12-15 10:00:00           2.0         56.0
12 2025-12-15 11:00:00           3.0         99.0
13 2025-12-15 12:00:00           3.0        100.0
14 2025-12-15 13:00:00           3.0        100.0
15 2025-12-15 14:00:00           3.0        100.0
16 2025-12-15 15:00:00           2.0         61.0
17 2025-12-15 16:00:00           3.0        100.0
18 2025-12-15 17:00:00           3.0

In [189]:
df.head()

,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus
1,2020-07-31 10:30:00,7,513.0,165.0,348.0
2,2020-07-31 11:00:00,7,594.0,134.0,460.0
3,2020-07-31 11:30:00,7,1286.0,147.0,1139.0
4,2020-07-31 12:00:00,7,2038.0,989.0,1049.0
5,2020-07-31 12:30:00,7,2052.0,451.0,1601.0


In [190]:
# Ensure data is sorted by date and time
df = df.sort_values(by=['timestamp'])
df.head()

,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus
1,2020-07-31 10:30:00,7,513.0,165.0,348.0
2,2020-07-31 11:00:00,7,594.0,134.0,460.0
3,2020-07-31 11:30:00,7,1286.0,147.0,1139.0
4,2020-07-31 12:00:00,7,2038.0,989.0,1049.0
5,2020-07-31 12:30:00,7,2052.0,451.0,1601.0


In [192]:
df_with_index = df.set_index('timestamp')

In [283]:
df_with_index.head(30)

,month,solar_energy_exported,total_home_usage,solar_surplus
timestamp,,,,
2020-07-31 10:30:00,7,513.0,165.0,348.0
2020-07-31 11:00:00,7,594.0,134.0,460.0
2020-07-31 11:30:00,7,1286.0,147.0,1139.0
2020-07-31 12:00:00,7,2038.0,989.0,1049.0
2020-07-31 12:30:00,7,2052.0,451.0,1601.0
2020-07-31 13:00:00,7,2081.0,162.0,1919.0
2020-07-31 13:30:00,7,2031.0,144.0,1887.0
2020-07-31 14:00:00,7,1994.0,179.0,1815.0
2020-07-31 14:30:00,7,1913.0,160.0,1753.0


In [396]:
# Get min and max dates
min_date=min(df.timestamp).date()
max_date=max(df.timestamp).date()
print(min_date,max_date)

2020-07-31 2025-12-09


In [406]:
# Get historiscal weather forecast by hour
from pathlib import Path
weather_filename = "weather_{min_date}_to_{max_date}.csv".format(min_date=min_date, max_date=max_date)
weather_file = Path("./{weather_filename}".format(weather_filename=weather_filename))
hourly_df = None
if weather_file.is_file():
    print("File '{weather_filename}' found, lodaing data".format(weather_filename=weather_filename))
    hourly_df = pd.read_csv(weather_filename, sep='\t', encoding='utf-8')
else:
    print("File '{weather_filename}' NOT found, calling meteo data API".format(weather_filename=weather_filename))
    hourly_df = get_historical_weather(latitude, longitude, min_date, max_date)

File 'weather_2020-07-31_to_2025-12-09.csv' found, lodaing data


In [407]:
hourly_df.head(24)

,timestamp,weather_code,cloud_cover
0,2020-07-30 23:00:00,0.0,19.0
1,2020-07-31 00:00:00,0.0,11.0
2,2020-07-31 01:00:00,3.0,99.0
3,2020-07-31 02:00:00,51.0,99.0
4,2020-07-31 03:00:00,3.0,94.0
5,2020-07-31 04:00:00,3.0,94.0
6,2020-07-31 05:00:00,2.0,57.0
7,2020-07-31 06:00:00,2.0,62.0
8,2020-07-31 07:00:00,1.0,35.0
9,2020-07-31 08:00:00,1.0,26.0


In [408]:
# Save the weather historical data
if not weather_file.is_file():
    #now=pd.Timestamp.now().strftime('%Y-%m-%d_%H%M%S')
    #filename = "{now}_weather_{min_date}_to_{max_date}.csv".format(now=now, min_date=min_date, max_date=max_date)
    hourly_df.to_csv(weather_filename, sep='\t', encoding='utf-8', index=False, header=True)
    print('Saved weather data to {filename}'.format(filename=weather_filename))

In [250]:
# merge energy data with weather historical data
df_merge=pd.merge(df_with_index, hourly_df, how='left', on='timestamp')

In [255]:
df_merge.count()

timestamp                31227
month                    31227
solar_energy_exported    31227
total_home_usage         31227
solar_surplus            31227
weather_code             15704
cloud_cover              15704
dtype: int64

In [409]:
df_merge.head(2)

,index,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover
0,0,2020-07-31 10:30:00,7,513.0,165.0,348.0,0.0,15.0
1,1,2020-07-31 11:00:00,7,594.0,134.0,460.0,0.0,15.0


In [389]:
# Fill missing data at half-hours with that from the beginning of the corresponding hours
import numpy as np

df_missing = df_merge[ ( np.isnan(df_merge.weather_code) | np.isnan(df_merge.cloud_cover) ) ]

if df_missing.size>0:
    print('Filling missing weather data at half-hours...')
    df_merge = df_merge.reset_index()  # make sure indexes pair with number of rows
    for idx, row in df_merge.iterrows():
        
        if row['timestamp'].minute==30:
            source_idx = ( idx-1 if idx > 0 else idx+1 )
            if (np.isnan(row['weather_code'])):
                df_merge.at[ idx, 'weather_code'] = df_merge.loc[source_idx]['weather_code']
            if (np.isnan(row['cloud_cover'])):
                df_merge.at[ idx, 'cloud_cover'] = df_merge.loc[source_idx]['cloud_cover']

In [390]:
df_merge.head(20)

,index,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover
0,0,2020-07-31 10:30:00,7,513.0,165.0,348.0,0.0,15.0
1,1,2020-07-31 11:00:00,7,594.0,134.0,460.0,0.0,15.0
2,2,2020-07-31 11:30:00,7,1286.0,147.0,1139.0,0.0,15.0
3,3,2020-07-31 12:00:00,7,2038.0,989.0,1049.0,1.0,30.0
4,4,2020-07-31 12:30:00,7,2052.0,451.0,1601.0,1.0,30.0
5,5,2020-07-31 13:00:00,7,2081.0,162.0,1919.0,2.0,69.0
6,6,2020-07-31 13:30:00,7,2031.0,144.0,1887.0,2.0,69.0
7,7,2020-07-31 14:00:00,7,1994.0,179.0,1815.0,2.0,68.0
8,8,2020-07-31 14:30:00,7,1913.0,160.0,1753.0,2.0,68.0
9,9,2020-07-31 15:00:00,7,1814.0,138.0,1676.0,2.0,66.0


In [410]:
# Check whether some weather data is missing
df_merge[ ( np.isnan(df_merge['weather_code']) | np.isnan(df_merge['cloud_cover']) )].size

0

In [411]:
df_merge

,index,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover
0,0,2020-07-31 10:30:00,7,513.0,165.0,348.0,0.0,15.0
1,1,2020-07-31 11:00:00,7,594.0,134.0,460.0,0.0,15.0
2,2,2020-07-31 11:30:00,7,1286.0,147.0,1139.0,0.0,15.0
3,3,2020-07-31 12:00:00,7,2038.0,989.0,1049.0,1.0,30.0
4,4,2020-07-31 12:30:00,7,2052.0,451.0,1601.0,1.0,30.0
...,...,...,...,...,...,...,...,...
31222,31222,2025-12-09 09:00:00,12,768.0,127.0,641.0,0.0,14.0
31223,31223,2025-12-09 09:30:00,12,808.0,118.0,690.0,0.0,14.0
31224,31224,2025-12-09 10:00:00,12,1036.0,132.0,904.0,0.0,0.0
31225,31225,2025-12-09 10:30:00,12,1216.0,870.0,346.0,0.0,0.0
